# Predicting next-month arrears
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on `billing_month`, choose by mean CV ROC AUC, holdout scored once).

**Feature timing.** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. around/after the outcome (87% arrears when 1 vs 1.4% when 0), so it is not known at prediction time and is excluded as leakage. `meter_ref` and `billing_month` are excluded by the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
is_hold=df.index.isin(hold_idx)
train=df[~is_hold]; hold=df[is_hold]
assert len(hold)==7536 and train.billing_month.max()<hold.billing_month.min()
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
print(len(train),len(hold),train[TARGET].mean(),hold[TARGET].mean())

## Candidates (named before any scoring)

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
CANDIDATES={
 'baseline_prior':Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'hgb_depth3':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'hgb_depth6':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=0))]),
}
list(CANDIDATES)

## Walk-forward CV on training rows (expanding window over billing months, 5 splits)

In [ ]:
months=np.sort(train.billing_month.unique())
blocks=np.array_split(months,6)   # block 0 = initial train; blocks 1..5 = successive validation folds
folds=[]
for k in range(1,6):
    tr=train.index[train.billing_month.isin(np.concatenate(blocks[:k]))]
    va=train.index[train.billing_month.isin(blocks[k])]
    folds.append((tr,va))
cv_scores={}
for name,est in CANDIDATES.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(df.loc[tr,FEATURES],df.loc[tr,TARGET])   # preprocessing fitted inside the fold
        s.append(roc_auc_score(df.loc[va,TARGET],m.predict_proba(df.loc[va,FEATURES])[:,1]))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); cv

In [ ]:
chosen=cv['mean'].idxmax(); print('chosen:',chosen)
final=clone(CANDIDATES[chosen]).fit(train[FEATURES],train[TARGET])
holdout_auc=roc_auc_score(hold[TARGET],final.predict_proba(hold[FEATURES])[:,1])   # scored once
print('holdout roc_auc',holdout_auc)

In [ ]:
evidence=dict(
 features=FEATURES,
 candidates=list(CANDIDATES),
 cv_folds=[[[int(i) for i in tr],[int(i) for i in va]] for tr,va in folds],
 cv_scores={k:[float(x) for x in v] for k,v in cv_scores.items()},
 chosen=chosen,
 holdout_rows=[int(i) for i in hold.index],
 holdout_scores={chosen:float(holdout_auc)},   # roc_auc, chosen candidate only
)
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. The holdout number is the headline; it was computed once, after the choice was made on CV.